
# Clean up the Beam account

**Runtime -> Run all.** Nothing is destroyed on the first pass: `DRY_RUN = True`
prints the plan and stops. Read it, set `DRY_RUN = False`, re-run the execute
cell.

### What this keeps and what it throws away

| | |
|---|---|
| **keep** | the `think-nano-weights` **volume** |
| **stop, but not delete** | every version of the app named in `PRESERVE_URL_FOR` |
| **delete** | everything else: the old `think-nano` app, the probes, all their versions |

The middle row is the one to understand before running this. Stopping a version
kills its containers -- which is exactly what frees one stuck holding an
in-flight request, the thing you are here to fix. *Deleting* every version of an
app risks taking the app with it, and the public hostname is derived from the
app, so that is what turns a redeploy into a new URL and a dead link on your own
site. Stopping gets you the fix; deleting gets you the fix plus a URL change you
did not ask for.

That split is the whole answer to *"is it easier to just delete it?"* -- for
deployments, yes, delete them all; `colab_beam_redeploy.ipynb` builds a fresh one
in a couple of minutes. For the volume, no. It holds 5.25 GiB that cost an 8.4 GiB
HuggingFace download, a bf16 export and an upload to put there, and none of the
symptoms you are chasing are caused by a volume that lists correctly. This
notebook proves it lists correctly *before* it deletes anything, and only offers
to delete it behind a typed confirmation at the very end.

### Why clutter is not just cosmetic

Every `beam deploy` leaves the previous version live with its own containers and
its own warm state. Renaming the app from `think-nano` to `bartholomew-iii`
created a whole second deployment rather than moving the first. So the account
now holds several independent things that:

- each hold their own containers and their own share of the account's GPU
  concurrency, which is what a new deployment has to schedule against;
- each keep billing while they idle inside `keep_warm_seconds`;
- make `beam logs` ambiguous -- the errors you are reading may belong to a
  version nothing is routed to any more.

Deleting all of them makes the next deploy the only deployment in the account,
which turns "which one is broken?" from a question into a non-question.

In [ ]:

# ---------------------------------------------------------------- configuration
# Nothing is destroyed while this is True.
DRY_RUN = True

# Deployment names to spare entirely -- not stopped, not deleted.
KEEP_DEPLOYMENTS = []

# Names whose versions get STOPPED but not DELETED.
#
# This is the difference between keeping your public URL and losing it. The
# hostname suffix is derived from the app, not from any one version, so an app
# that still exists redeploys to the same URL. Delete every version and you are
# gambling that the app record survives -- if it does not, the next deploy comes
# back on a different hostname and every link you have published, including the
# one on your own site, is dead until you update it.
#
# Stopping is enough for what you are actually fixing: it kills the containers,
# which is what frees one stuck on an in-flight request, and the redeploy brings
# up new ones regardless.
PRESERVE_URL_FOR = ['bartholomew-iii']

# Stop, then delete. Stopping alone ends the billing and settles "is an old
# version still serving?"; deleting also clears the dashboard. Set False to stop
# only -- reversible with `beam deployment start <id>`.
DELETE = True

# The volume this notebook protects. It is never touched except by the very last
# cell, which needs a typed confirmation.
VOLUME = 'think-nano-weights'
MODEL_TAG = 'Think.Unbounded-d32-v2mix-cont-pre1930-curriculum-c3-robust-v2'
# -----------------------------------------------------------------------------

import json, os, re, shutil, subprocess, sys, time
import urllib.error, urllib.request


def _stream(cmd, stdin_text=None, quiet=False, timeout=None):
    """Run cmd, echoing output into the notebook. Returns (exit_code, output).

    subprocess.check_call shows nothing in Colab -- the child inherits the
    kernel's file descriptor while the notebook only captures writes to
    IPython's redirected sys.stdout, so you get a bare `0`. Reading the child's
    pipe and print()ing it is what actually displays, and unlike `!cmd` it still
    lets us branch on the exit code.

    `timeout` kills the child after that many seconds. Some beam subcommands
    tail rather than return -- `beam logs` most of all -- and one of those in a
    loop is the difference between a cell that takes two seconds and a cell you
    have to interrupt.
    """
    import threading

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        stdin=subprocess.PIPE if stdin_text is not None else subprocess.DEVNULL,
        text=True, bufsize=1, errors='replace')
    if stdin_text is not None:
        proc.stdin.write(stdin_text); proc.stdin.flush(); proc.stdin.close()

    # A flag rather than the exit code: a killed child reports -9 on Linux but 1
    # on Windows, and this notebook should read the same either way.
    fired = []

    def _kill():
        fired.append(True)
        proc.kill()

    killer = threading.Timer(timeout, _kill) if timeout else None
    if killer:
        killer.daemon = True
        killer.start()
    out = []
    try:
        for line in proc.stdout:
            if not quiet:
                print(line, end='')
            out.append(line)
        code = proc.wait()
    finally:
        if killer:
            killer.cancel()
    if fired:
        out.append(f'\n[killed after {timeout}s -- this command does not return on its own]\n')
        if not quiet:
            print(out[-1], end='')
    return code, ''.join(out)


def run(cmd, check=True, stdin_text=None, quiet=False, timeout=None):
    code, out = _stream(cmd, stdin_text, quiet, timeout)
    if check and code != 0:
        raise subprocess.CalledProcessError(code, ' '.join(cmd), out)
    return out


# Beam draws its tables with box characters; older builds used ASCII pipes.
_RULES = '\u2502\u2503|'


def parse_table(text):
    """Beam's CLI table as a list of dicts. Deliberately forgiving.

    The table layout is not an API and has changed shape before, so every caller
    here keeps the raw text too: a parse that comes back empty degrades to "read
    it yourself above" rather than to a crash in the middle of a cleanup.
    """
    rows = []
    for line in text.splitlines():
        if not any(ch in line for ch in _RULES):
            continue
        cells = [c.strip() for c in re.split('[' + _RULES + ']', line.strip())]
        cells = [c for c in cells if c]
        if cells:
            rows.append(cells)
    if not rows:
        return []
    header = [re.sub(r'\W+', '_', h.lower()).strip('_') for h in rows[0]]
    parsed = []
    for cells in rows[1:]:
        if len(cells) != len(header):
            continue
        row = dict(zip(header, cells))
        if all(set(v) <= set('-\u2500\u2501 ') for v in row.values()):
            continue        # a horizontal rule that happened to have pipes
        parsed.append(row)
    return parsed


def field(row, *candidates):
    """Read a column by any of several plausible names, ignoring punctuation."""
    norm = {re.sub(r'[^a-z0-9]', '', str(k).lower()): v for k, v in row.items()}
    for c in candidates:
        key = re.sub(r'[^a-z0-9]', '', c.lower())
        if key in norm and norm[key] not in (None, ''):
            return str(norm[key])
    return ''


def beam_rows(args, label):
    """`beam <args>` as a list of dicts, preferring JSON if this CLI offers it."""
    code, out = _stream(['beam'] + args + ['--format', 'json'], quiet=True)
    if code == 0 and '[' in out and ']' in out:
        try:
            data = json.loads(out[out.index('['):out.rindex(']') + 1])
            if isinstance(data, list):
                print(f'{label}: {len(data)} row(s), read as JSON')
                for row in data:
                    print('  ' + json.dumps(row))
                return data, out
        except Exception:
            pass
    code, out = _stream(['beam'] + args)
    return (parse_table(out) if code == 0 else []), out

from google.colab import userdata

BEAM_TOKEN = userdata.get('BEAM_TOKEN')
assert BEAM_TOKEN, 'Set BEAM_TOKEN in Colab secrets (key icon, left sidebar)'
print('config ok -- DRY_RUN =', DRY_RUN)

## Install and authenticate

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'beam-client'])

# `beam configure` asks before overwriting an existing context and a notebook has
# no stdin to answer with, so answer it explicitly. `beam machine list` is the
# real check: if that returns, the credentials are good.
run(['beam', 'configure', 'default', '--token', BEAM_TOKEN],
    stdin_text='y\n', check=False)
run(['beam', 'machine', 'list'])


## What this CLI can actually do

Printed rather than assumed. `beam-client` moves commands between releases, and
a cleanup notebook that guesses a subcommand name and gets an unrecognised-command
error halfway through is worse than one that shows you the surface first. If
anything below is missing from your version, the corresponding cell degrades to
printing what it found instead of acting on it.

In [ ]:

for group in ([], ['deployment'], ['volume'], ['pod'], ['task']):
    print('=' * 70)
    print('beam ' + ' '.join(group + ['--help']))
    print('=' * 70)
    _stream(['beam'] + group + ['--help'])
    print()


## Inventory

Everything the account is currently holding. This is the picture the dashboard
makes hard to read: deployments and their versions, any pods, the volumes, and
whether tasks are sitting in a queue.

A queue that is not empty is worth noticing on its own -- with
`tasks_per_container = 1`, requests stacked behind a task that never finished are
indistinguishable, from a browser, from a model that is broken.

In [ ]:

DEPLOYMENTS, dep_raw = beam_rows(['deployment', 'list'], 'deployments')
print()
POOLS, pod_raw = beam_rows(['pod', 'list'], 'pods')
print()
VOLUMES, vol_raw = beam_rows(['volume', 'list'], 'volumes')
print()
TASKS, task_raw = beam_rows(['task', 'list'], 'tasks')

print()
print('=' * 70)
print(f'{len(DEPLOYMENTS)} deployment(s), {len(POOLS)} pod(s), '
      f'{len(VOLUMES)} volume(s), {len(TASKS)} task(s)')

names = sorted({field(d, 'name', 'app_name', 'stub_name') for d in DEPLOYMENTS} - {''})
print('deployment names:', ', '.join(names) or '(none parsed -- read the table above)')

pending = [t for t in TASKS
           if field(t, 'status', 'state').upper() in ('PENDING', 'RUNNING', 'QUEUED')]
if pending:
    print(f'\nNOTE  {len(pending)} task(s) are PENDING/RUNNING. If these belong to a '
          'deployment\n      you are about to delete they go with it; if they belong to '
          'the one you\n      are keeping, they are ahead of every request in the queue.')


## Prove the weights are intact -- before deleting anything

The order matters. Confirming what is on the volume while the deployments still
exist means that if this cell reports the checkpoint missing, you have learned
the actual cause of the outage and can stop, rather than having just deleted the
evidence.

Three things have to be there, and the container reads all three by path:

```
<VOLUME>/<MODEL_TAG>/model_XXXXXX.pt
<VOLUME>/<MODEL_TAG>/meta_XXXXXX.json
<VOLUME>/<MODEL_TAG>/tokenizer/tokenizer.pkl
<VOLUME>/pre1930-companion.txt          <- the persona, optional but wanted
```

In [ ]:

_c_root, root_listing = _stream(['beam', 'ls', VOLUME])
print()
_c_ckpt, ckpt_listing = _stream(['beam', 'ls', f'{VOLUME}/{MODEL_TAG}'])
print()
_c_tok, tok_listing = _stream(['beam', 'ls', f'{VOLUME}/{MODEL_TAG}/tokenizer'])

models = sorted(int(m) for m in re.findall(r'model_(\d+)\.pt', ckpt_listing))
metas = sorted(int(m) for m in re.findall(r'meta_(\d+)\.json', ckpt_listing))
has_tokenizer = 'tokenizer.pkl' in tok_listing
has_persona = 'pre1930-companion.txt' in root_listing

STEP = max(set(models) & set(metas)) if (models and metas) else None
WEIGHTS_OK = STEP is not None and has_tokenizer

print()
print('=' * 70)
print(f'model_*.pt      {models or "MISSING"}')
print(f'meta_*.json     {metas or "MISSING"}')
print(f'tokenizer.pkl   {"present" if has_tokenizer else "MISSING"}')
print(f'persona file    {"present" if has_persona else "missing (deployment will serve without one)"}')
print(f'usable step     {STEP}')
print()
if WEIGHTS_OK:
    print('WEIGHTS OK -- the volume is complete. Deleting deployments is safe and')
    print('the redeploy notebook will find everything it needs.')
else:
    print('WEIGHTS INCOMPLETE. This alone would explain the container errors: the')
    print('loader raises FileNotFoundError in on_start and every request afterwards')
    print('fails. Do not delete the volume. Re-run colab_beam_deploy.ipynb with')
    print('FORCE_REPREP = True to rebuild it before doing anything else here.')


## The plan

Printed in full before anything runs. Deletion is by id, not by name, so a
deployment whose row failed to parse is reported as un-actionable rather than
matched by a guess.

In [ ]:

# The subcommands this plan depends on, confirmed rather than assumed. `beam
# logs` already turned out to take options where the runbook expected a
# positional, and finding that out mid-cleanup would be worse.
_c, dep_help = _stream(['beam', 'deployment', '--help'], quiet=True, timeout=30)
for verb in ('stop', 'delete'):
    if not re.search(r'^\s*' + verb + r'\b', dep_help, re.M):
        print(f'WARN  `beam deployment {verb}` is not in this CLI\'s help. The plan below')
        print(f'      may not run. `beam deployment --help` output:')
        print(dep_help)
        break
else:
    print('beam deployment stop / delete both present.')
print()

targets, unparsed = [], 0
for d in DEPLOYMENTS:
    did = field(d, 'id', 'deployment_id', 'stub_id')
    name = field(d, 'name', 'app_name', 'stub_name')
    if not did:
        unparsed += 1
        continue
    if name in KEEP_DEPLOYMENTS:
        action = 'KEEP'
    elif name in PRESERVE_URL_FOR:
        action = 'stop only (keeps the URL)'
    else:
        action = 'stop + delete' if DELETE else 'stop'
    targets.append(dict(id=did, name=name, action=action,
                        version=field(d, 'version', 'v'),
                        active=field(d, 'active', 'status', 'state')))

print(f'{"name":<22}{"ver":<5}{"active":<9}{"id":<38}action')
print('-' * 100)
for t in targets:
    print(f'{t["name"][:21]:<22}{t["version"]:<5}{t["active"][:8]:<9}{t["id"]:<38}{t["action"]}')

doomed = [t for t in targets if t['action'] != 'KEEP']
to_delete = [t for t in targets if t['action'].startswith('stop + delete')]
preserved = sorted({t['name'] for t in targets if t['name'] in PRESERVE_URL_FOR})
print()
print(f'{len(doomed)} deployment(s) to act on: {len(to_delete)} deleted, '
      f'{len(doomed) - len(to_delete)} stopped only.')
if preserved:
    print(f'App(s) kept so the URL survives: {", ".join(preserved)}')
    print('  Their containers stop -- which is the point, it frees one stuck on an')
    print('  in-flight request -- but the app is not deleted, so the redeploy should')
    print('  come back on the same hostname.')
if unparsed:
    print(f'{unparsed} row(s) had no readable id -- handle those by hand from the table above.')
print(f'The {VOLUME} volume is NOT touched by the cell below.')
print()
print('DRY RUN -- nothing will happen.' if DRY_RUN
      else 'ARMED -- the next cell will really do this.')


## Execute

Set `DRY_RUN = False` in the configuration cell, re-run it, then run this.

Stop before delete on purpose: a stop takes effect immediately, so a delete that
is refused (because a container is still draining, say) still leaves the
deployment inert rather than half-handled.

In [ ]:

if 'doomed' not in dir():
    raise SystemExit('Run the plan cell above first -- this cell acts on what it built.')

if DRY_RUN:
    print('DRY_RUN is True -- nothing done. Set it to False and re-run.')
elif not doomed:
    print('Nothing to do.')
else:
    failures = []
    for t in doomed:
        print(f'--- {t["name"]} v{t["version"]} ({t["id"]})  [{t["action"]}]')
        code, _ = _stream(['beam', 'deployment', 'stop', t['id']], timeout=120)
        if code != 0:
            print('    stop failed (often just "already stopped")')
        if t['action'].startswith('stop + delete'):
            code, out = _stream(['beam', 'deployment', 'delete', t['id']], timeout=120)
            if code != 0:
                failures.append((t, out.strip().splitlines()[-1:] or ['']))
        print()

    print('=' * 70)
    print(f'{len(doomed) - len(failures)}/{len(doomed)} handled.')
    for t, msg in failures:
        print(f'  FAILED  {t["name"]} {t["id"]}: {msg[0] if msg else ""}')
    if failures:
        print('\nDelete those by hand from the dashboard, or leave them stopped --')
        print('a stopped deployment costs nothing and cannot answer a request.')


## Confirm the account is clean

In [ ]:

print('=' * 70)
print('deployments now')
print('=' * 70)
_stream(['beam', 'deployment', 'list'])
print()
print('=' * 70)
print('volumes now  (think-nano-weights must still be here)')
print('=' * 70)
_stream(['beam', 'volume', 'list'])
print()
_stream(['beam', 'ls', f'{VOLUME}/{MODEL_TAG}'])


## Optional: delete the volume too

Only if you have decided to rebuild the whole thing from HuggingFace. This throws
away the 5.25 GiB export; getting it back is `colab_beam_deploy.ipynb` with
`FORCE_REPREP = True`, which is a download, an export and an upload -- call it
forty minutes of Colab time, none of it interactive.

Worth doing if the check above reported the checkpoint incomplete or the step
numbers are not what you expect. Not worth doing to tidy up.

To arm it, replace the empty string with the volume name exactly.

In [ ]:

CONFIRM = ''      # type the volume name here to arm this cell

if CONFIRM != VOLUME:
    print(f'Not armed. To delete the volume set CONFIRM = {VOLUME!r}.')
    print('You almost certainly do not want to.')
else:
    print('Deleting the volume. Rebuild it with colab_beam_deploy.ipynb, FORCE_REPREP = True.')
    _stream(['beam', 'volume', 'delete', VOLUME])


## Next

1. If `WEIGHTS OK` printed above, go straight to **`colab_beam_redeploy.ipynb`**.
   A clean account plus a complete volume is the state that notebook is built for.
2. If you want to know *what* was failing before you replace it, run
   **`colab_beam_debug.ipynb`** first -- but note it needs a live deployment to
   interrogate, so run it **before** this notebook, not after.
3. **Check the URL the redeploy prints against the one on your site.** With
   `PRESERVE_URL_FOR` covering the app, it should come back unchanged -- but
   "should" is doing real work in that sentence, because the hostname suffix is
   not derivable from anything the CLI exposes, so this notebook cannot promise
   it. Read the URL off the redeploy output and compare.

4. **Drop any `-vN` from the link on your site.** A versioned URL is pinned to
   one deployment version, and this notebook has just stopped the versions that
   existed. `https://<name>-<suffix>.app.beam.cloud`, with no `-v2`, is the form
   that follows each new deploy; that is what belongs in
   `<meta name="api-base">`.